# Validate the Premier League appearance pipeline

This notebook inspects the ignored league-wide outputs created by `scripts/build_league_data.py`. It checks coverage, position mapping, unresolved appearances and the high-confidence centre-back population before any positional template is calculated.

In [ ]:
from pathlib import Path

import pandas as pd

search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next(
    (root for root in search_roots if (root / 'data' / 'processed').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open this notebook from inside the project directory')

PROCESSED_ROOT = PROJECT_ROOT / 'data' / 'processed'
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 200)
PROJECT_ROOT

In [ ]:
eligible_appearances = pd.read_parquet(
    PROCESSED_ROOT / 'premier_league_eligible_appearances.parquet'
)
eligible_actions = pd.read_parquet(
    PROCESSED_ROOT / 'premier_league_eligible_positional_actions.parquet'
)
centre_back_appearances = pd.read_parquet(
    PROCESSED_ROOT / 'premier_league_centre_back_appearances.parquet'
)
quality_checks = pd.read_csv(
    PROCESSED_ROOT / 'premier_league_pipeline_quality_checks.csv'
)

print(f'Eligible appearances: {len(eligible_appearances):,}')
print(f'Eligible direct actions: {len(eligible_actions):,}')
print(f'High-confidence centre-back appearances: {len(centre_back_appearances):,}')

## Reproducible quality checks

In [ ]:
quality_checks

## Team and position coverage

In [ ]:
team_coverage = (
    eligible_appearances
    .groupby('team_name')
    .agg(
        eligible_appearances=('player_id', 'size'),
        eligible_players=('player_id', 'nunique'),
        available_matches=('match_id', 'nunique'),
        mapped_appearances=(
            'mapping_status',
            lambda values: values.eq('mapped').sum(),
        ),
    )
    .sort_index()
)
team_coverage

In [ ]:
position_coverage = (
    eligible_appearances
    .groupby(
        ['position_group', 'standardised_position'],
        dropna=False,
    )
    .size()
    .rename('appearance_count')
    .reset_index()
    .sort_values(['position_group', 'appearance_count'], ascending=[True, False])
)
position_coverage

## Unresolved appearances

These appearances remain outside all high-confidence positional templates. In this version they should be early substitutes whose initial team-sheet slot is `0`.

In [ ]:
unresolved_columns = [
    'match_date',
    'match_description',
    'team_name',
    'player_name',
    'formation_code',
    'lineup_slot',
    'is_starter',
    'minute_on',
    'minutes_played',
    'mapping_status',
]

unresolved_appearances = (
    eligible_appearances.loc[
        eligible_appearances['mapping_status'].ne('mapped'),
        unresolved_columns,
    ]
    .sort_values(['team_name', 'match_date', 'player_name'])
    .reset_index(drop=True)
)
unresolved_appearances

## High-confidence centre-back population

In [ ]:
centre_backs_by_team = (
    centre_back_appearances
    .groupby('team_name')
    .agg(
        centre_back_appearances=('player_id', 'size'),
        centre_back_players=('player_id', 'nunique'),
        available_matches=('match_id', 'nunique'),
    )
    .sort_values('centre_back_appearances', ascending=False)
)
centre_backs_by_team

In [ ]:
centre_backs_by_player = (
    centre_back_appearances
    .groupby(['team_name', 'player_id', 'player_name'])
    .agg(
        eligible_appearances=('match_id', 'nunique'),
        total_minutes=('minutes_played', 'sum'),
        direct_actions=('action_count', 'sum'),
    )
    .reset_index()
    .sort_values(['eligible_appearances', 'team_name'], ascending=[False, True])
)
centre_backs_by_player

## Forest regression check

The general pipeline should reproduce the original Forest defender sample: 144 eligible appearances and 9,739 direct actions.

In [ ]:
forest_defender_appearances = eligible_appearances.loc[
    eligible_appearances['team_name'].str.contains('Forest', case=False, na=False)
    & eligible_appearances['squad_position'].eq('Defender')
].copy()
forest_keys = forest_defender_appearances[
    ['match_id', 'team_id', 'player_id']
]
forest_defender_actions = eligible_actions.merge(
    forest_keys,
    on=['match_id', 'team_id', 'player_id'],
    how='inner',
)

assert len(forest_defender_appearances) == 144
assert len(forest_defender_actions) == 9_739

pd.Series(
    {
        'Forest eligible defender appearances': len(forest_defender_appearances),
        'Forest eligible defender actions': len(forest_defender_actions),
        'Forest high-confidence centre-back appearances': len(
            centre_back_appearances.loc[
                centre_back_appearances['team_name'].str.contains(
                    'Forest',
                    case=False,
                    na=False,
                )
            ]
        ),
    },
    name='value',
)